In [1]:
from langchain_milvus import Milvus
from langchain_community.embeddings import DashScopeEmbeddings
from dotenv import load_dotenv
load_dotenv()

C:\Users\q2005\AppData\Local\Temp\ipykernel_15876\2689591876.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import DashScopeEmbeddings


True

In [2]:
# 向量化模型
embed_fn = DashScopeEmbeddings(model='text-embedding-v4')

In [3]:
 from langchain_milvus import Milvus

db = Milvus(
     embedding_function=embed_fn,
     collection_name="my_collection",
     auto_id=True,
     index_params={"index_type": "HNSW", "metric_type": "L2"},
     connection_args={"alias": "default"},
     drop_old=True
 )

D:\Anaconda3\envs\myrag\lib\site-packages\langchain_milvus\vectorstores\milvus.py:408: RuntimeWarning: coroutine 'AsyncMilvusClient._get_connection' was never awaited
  self.drop()


In [4]:
# 删掉内容
# db.drop()

In [5]:
 texts = [
     "Milvus 是开源的向量数据库，支持高维向量的快速检索",
     "LangChain 可以集成 Milvus 实现 RAG 应用的向量存储",
     "RAG 检索增强生成能解决大模型知识过时的问题",
     "Milvus 支持多种索引类型，如 IVF_FLAT、HNSW 等"
 ]

In [6]:
 db.add_texts(texts=texts)

D:\Anaconda3\envs\myrag\lib\site-packages\langchain_milvus\vectorstores\milvus.py:580: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  self._col_cache = Collection(self.collection_name, using=self.alias)


ConnectionNotExistException: <ConnectionNotExistException: (code=1, message=should create connection first.)>

In [ ]:
 db.similarity_search_with_relevance_scores(query="Milvus是什么？", 
                                            k=4, 
                                            kwargs={"score_threshold": 0.75})

In [ ]:
from pymilvus import MilvusClient
from langchain_community.embeddings import DashScopeEmbeddings
import os
from dotenv import load_dotenv

load_dotenv()

# 模型
embeddings = DashScopeEmbeddings(
    model="text-embedding-v4",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")
)

# 连接 Milvus
client = MilvusClient(uri="http://localhost:19530")
collection_name = "test_rag_final"

# 删掉旧集合
if client.has_collection(collection_name):
    client.drop_collection(collection_name)


client.create_collection(
    collection_name=collection_name,
    dimension=1024,  
)

# 数据
texts = [
    "Milvus 是一个向量数据库",
    "LangChain 用于构建 RAG 应用",
    "我正在学习 AI 知识库"
]

# 生成向量
vectors = embeddings.embed_documents(texts)

# 插入
data = [
    {"id": i, "vector": vectors[i], "text": texts[i]}
    for i in range(len(texts))
]
client.insert(collection_name=collection_name, data=data)

print("成功写入 Milvus！！！")

In [ ]:
# 搜索功能
def search_similar(query, collection_name="test_rag_final", k=3):
    query_vector = embed_fn.embed_query(query)
    results = client.search(
        collection_name=collection_name,
        data=[query_vector],
        limit=k,
        output_fields=["text"]
    )
    
    print(f"查询: {query}\n")
    for i, result in enumerate(results[0], 1):
        print(f"{i}. 相似度: {result['distance']:.4f}")
        print(f"   内容: {result['entity']['text']}\n")
    
    return results

# 测试搜索
search_similar("什么是Milvus？")